In [17]:
import sys
import json
import yaml
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")

print("Python:", sys.executable)
print("프로젝트:", ROOT)

PLAN_ROOT = (
    ROOT.parent
    / "datasets/hemorrhage/v1/model_ready/nnunet"
    / "nnUNet_preprocessed/Dataset001_ICHBinary"
)

# nnU-Net reader 설정 수정
for name in [
    "nnUNetResEncUNetMPlans.json",
    "ICHPlansResEncMV1.json",
    "ICHPlansResEncMAttentionV1.json",
]:
    path = PLAN_ROOT / name
    data = json.loads(path.read_text(encoding="utf-8"))
    data["image_reader_writer"] = "NibabelIO"
    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print("수정:", path.name)

# SegResNet worker를 0으로 설정
for experiment in ["H-S10", "H-S20"]:
    path = (
        ROOT
        / f"configs/experiments/hemorrhage/{experiment}.yaml"
    )
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    config["data"]["num_workers"] = 0
    path.write_text(
        yaml.safe_dump(
            config,
            allow_unicode=True,
            sort_keys=False,
        ),
        encoding="utf-8",
    )
    print("수정:", path.name)

Python: /home/stu03/workspace/model/.venv/bin/python
프로젝트: /home/stu03/workspace/model
수정: nnUNetResEncUNetMPlans.json
수정: ICHPlansResEncMV1.json
수정: ICHPlansResEncMAttentionV1.json
수정: H-S10.yaml
수정: H-S20.yaml


In [20]:
import json
import yaml
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")

PLAN_ROOT = (
    ROOT.parent
    / "datasets/hemorrhage/v1/model_ready/nnunet"
    / "nnUNet_preprocessed/Dataset001_ICHBinary"
)

print("=== nnU-Net reader 확인 ===")

for name in [
    "nnUNetResEncUNetMPlans.json",
    "ICHPlansResEncMV1.json",
    "ICHPlansResEncMAttentionV1.json",
]:
    path = PLAN_ROOT / name
    data = json.loads(path.read_text(encoding="utf-8"))
    reader = data.get("image_reader_writer")

    print(f"{name}: {reader}")
    assert reader == "NibabelIO"

print()
print("=== SegResNet worker 확인 ===")

for experiment in ["H-S10", "H-S20"]:
    path = (
        ROOT
        / f"configs/experiments/hemorrhage/{experiment}.yaml"
    )
    config = yaml.safe_load(path.read_text(encoding="utf-8"))
    workers = config["data"]["num_workers"]

    print(f"{experiment}: num_workers={workers}")
    assert workers == 0

print()
print("모든 수정 사항이 정상적으로 적용돼 있습니다.")

=== nnU-Net reader 확인 ===
nnUNetResEncUNetMPlans.json: NibabelIO
ICHPlansResEncMV1.json: NibabelIO
ICHPlansResEncMAttentionV1.json: NibabelIO

=== SegResNet worker 확인 ===
H-S10: num_workers=0
H-S20: num_workers=0

모든 수정 사항이 정상적으로 적용돼 있습니다.


In [19]:
# 실행 전 점검

import subprocess

result = subprocess.run(
    [
        sys.executable,
        "scripts/run_hemorrhage_variants.py",
        "--dry-run",
        "--only",
        "H-N20",
        "H-S10",
        "H-S20",
        "--gpu",
        "0",
    ],
    cwd=ROOT,
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)

data_root=/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet
segresnet_root=/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/segresnet
manifest=/home/stu03/workspace/datasets/hemorrhage/v1/metadata/common/06_preprocessing_manifest.csv
H-N20: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.nnunet.runner --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-N20.yaml --data-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet --results-root /home/stu03/workspace/model/runs/nnunet
H-S10: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.models.hemorrhage.segresnet.train --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-S10.yaml
H-S20: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.models.hemorrhage.segresnet.train --config /home/stu03/workspace/model/configs/experiments/hemorrhage/H-S20.yaml



In [11]:
import os
import sys
import time
import signal
import subprocess
from pathlib import Path

ROOT = Path("/home/stu03/workspace/model")
RUN_DIR = ROOT / "runs/hemorrhage-variants"
PID_FILE = RUN_DIR / "batch.pid"
LOG_FILE = RUN_DIR / "launcher.log"

RUN_DIR.mkdir(parents=True, exist_ok=True)

# 기존 배치가 실행 중이면 종료
if PID_FILE.exists():
    old_pid = int(PID_FILE.read_text().strip())

    try:
        os.kill(old_pid, 0)
        print(f"기존 배치 종료 중: PID={old_pid}")
        os.killpg(old_pid, signal.SIGTERM)
        time.sleep(3)
    except ProcessLookupError:
        print("기존 PID는 이미 종료된 상태입니다.")

# 실행 환경 설정
environment = os.environ.copy()
environment["nnUNet_n_proc_DA"] = "0"
environment["nnUNet_def_n_proc"] = "1"

# 새 배치 실행
with LOG_FILE.open("a", encoding="utf-8") as log:
    process = subprocess.Popen(
        [
            sys.executable,
            "scripts/run_hemorrhage_variants.py",
            "--gpu",
            "0",
        ],
        cwd=ROOT,
        env=environment,
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

PID_FILE.write_text(str(process.pid))

print(f"새 배치 실행 PID: {process.pid}")
print("nnU-Net worker: 2")
print("실행 순서: H-N10 → H-N20 → H-S10 → H-S20")
print(f"로그: {LOG_FILE}")

기존 배치 종료 중: PID=73839
새 배치 실행 PID: 76386
nnU-Net worker: 2
실행 순서: H-N10 → H-N20 → H-S10 → H-S20
로그: /home/stu03/workspace/model/runs/hemorrhage-variants/launcher.log


In [12]:
# 실시간 진행률 표시
BATCH_STATUS = RUN_DIR / "batch_status.json"

PROGRESS_FILES = {
    "H-N10": (
        ROOT / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansResEncMV1__3d_fullres/"
        "fold_0/progress.json"
    ),
    "H-N20": (
        ROOT / "runs/nnunet/Dataset001_ICHBinary/"
        "nnUNetTrainerICHBaseline__ICHPlansResEncMAttentionV1__3d_fullres/"
        "fold_0/progress.json"
    ),
    "H-S10": ROOT / "runs/segresnet/H-S10/progress.json",
    "H-S20": ROOT / "runs/segresnet/H-S20/progress.json",
}


def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def format_time(seconds):
    if seconds is None:
        return "-"

    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)

    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


while True:
    clear_output(wait=True)

    batch = read_json(BATCH_STATUS)
    batch_items = batch.get("experiments", {})
    rows = []

    for experiment, path in PROGRESS_FILES.items():
        progress = read_json(path)
        batch_item = batch_items.get(experiment, {})

        rows.append({
            "실험": experiment,
            "상태": progress.get(
                "status",
                batch_item.get("status", "waiting"),
            ),
            "Epoch": (
                f"{progress.get('current_epoch', 0)}"
                f"/{progress.get('total_epochs', '-')}"
            ),
            "진행률(%)": progress.get("progress_percent", 0),
            "ETA": format_time(progress.get("eta_seconds")),
            "Loss": progress.get("train_loss"),
            "Dice": progress.get(
                "dice",
                progress.get("pseudo_dice"),
            ),
            "Recall": progress.get(
                "recall",
                progress.get("pseudo_recall"),
            ),
            "Best Composite": progress.get("best_composite"),
            "GPU 메모리(GB)": progress.get("gpu_max_memory_gb"),
        })

    print("전체 상태:", batch.get("status", "starting"))
    print("현재 실험:", batch.get("current", "-"))
    print()

    display(pd.DataFrame(rows))

    if batch.get("status") in {"completed", "failed"}:
        print("배치 실행이 종료됐습니다.")
        break

    time.sleep(10)

전체 상태: failed
현재 실험: H-N10



,실험,상태,Epoch,진행률(%),ETA,Loss,Dice,Recall,Best Composite,GPU 메모리(GB)
0,H-N10,early_stopped,229/1000,22.9,05:12:11,None,None,None,0.842841,None
1,H-N20,waiting,0/-,0.0,-,None,None,None,NaN,None
2,H-S10,waiting,0/-,0.0,-,None,None,None,NaN,None
3,H-S20,waiting,0/-,0.0,-,None,None,None,NaN,None


배치 실행이 종료됐습니다.


In [8]:
import os
import signal
import time
from pathlib import Path

RUN_DIR = Path(
    "/home/stu03/workspace/model/runs/hemorrhage-variants"
)

pid = int((RUN_DIR / "batch.pid").read_text().strip())
os.killpg(pid, signal.SIGTERM)
time.sleep(5)

print("종료 요청 완료:", pid)

종료 요청 완료: 73839


In [ ]:
import os
import signal
from pathlib import Path

pid_file = Path(
    "/home/stu03/workspace/model/"
    "runs/hemorrhage-variants/batch.pid"
)

pid = int(pid_file.read_text().strip())
os.killpg(pid, signal.SIGTERM)

print("종료 요청:", pid)

종료 요청: 70179


In [ ]:
# 현재 로그 확인
batch = read_json(BATCH_STATUS)
current = batch.get("current")

if current:
    log_file = RUN_DIR / "logs" / f"{current}.log"

    if log_file.exists():
        lines = log_file.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()

        print("\n".join(lines[-50:]))
    else:
        print("로그 파일 생성 대기 중")
else:
    print("현재 실행 중인 실험이 없습니다.")